In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("../data/raw/GlobalWeatherRepository.csv")
df.head()


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [4]:
numeric_cols = [
    'latitude', 'longitude',
    'temperature_celsius', 'feels_like_celsius',
    'humidity', 'cloud',
    'wind_kph', 'gust_kph', 'wind_degree',
    'pressure_mb',
    'precip_mm',
    'visibility_km',
    'uv_index',
    'air_quality_Carbon_Monoxide',
    'air_quality_Ozone',
    'air_quality_Nitrogen_dioxide',
    'air_quality_Sulphur_dioxide',
    'air_quality_PM2.5',
    'air_quality_PM10',
    'air_quality_us-epa-index',
    'air_quality_gb-defra-index'
]


In [5]:
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [6]:
df = df.drop_duplicates()


In [7]:
(df[numeric_cols].isnull().sum() / len(df)) * 100


latitude                        0.0
longitude                       0.0
temperature_celsius             0.0
feels_like_celsius              0.0
humidity                        0.0
cloud                           0.0
wind_kph                        0.0
gust_kph                        0.0
wind_degree                     0.0
pressure_mb                     0.0
precip_mm                       0.0
visibility_km                   0.0
uv_index                        0.0
air_quality_Carbon_Monoxide     0.0
air_quality_Ozone               0.0
air_quality_Nitrogen_dioxide    0.0
air_quality_Sulphur_dioxide     0.0
air_quality_PM2.5               0.0
air_quality_PM10                0.0
air_quality_us-epa-index        0.0
air_quality_gb-defra-index      0.0
dtype: float64

### Missing Value Analysis
No missing values were detected in the dataset. Therefore, no imputation required.


### Outlier Handling

In [8]:
def iqr_capping(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series.clip(lower, upper)


In [9]:
outlier_columns = [
    'temperature_celsius',
    'wind_kph',
    'gust_kph',
    'pressure_mb',
    'visibility_km'
]

for col in outlier_columns:
    df[col] = iqr_capping(df[col])


### Encoding

In [10]:
df.select_dtypes(include='object').columns


Index(['country', 'location_name', 'timezone', 'last_updated',
       'condition_text', 'wind_direction', 'sunrise', 'sunset', 'moonrise',
       'moonset', 'moon_phase'],
      dtype='object')

In [11]:
df_encoded = pd.get_dummies(df, columns=['condition_text', 'wind_direction'], drop_first=True)


### Feature Engineering

In [12]:
df['last_updated'] = pd.to_datetime(df['last_updated'])

df['year'] = df['last_updated'].dt.year
df['month'] = df['last_updated'].dt.month
df['day'] = df['last_updated'].dt.day
df['day_of_week'] = df['last_updated'].dt.dayofweek


#### Seasonal Feature

In [13]:
season_map = {
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring', 4:'Spring', 5:'Spring',
    6:'Summer', 7:'Summer', 8:'Summer',
    9:'Autumn', 10:'Autumn', 11:'Autumn'
}

df['season'] = df['month'].map(season_map)


#### Temperature Difference Feature

In [14]:
df['temperature_difference'] = df['temperature_celsius'] - df['feels_like_celsius']


In [15]:
print("Final Dataset Shape:", df.shape)
print("Date Range:", df['last_updated'].min(), "to", df['last_updated'].max())
print("Number of Countries:", df['country'].nunique())


Final Dataset Shape: (122966, 47)
Date Range: 2024-05-16 01:45:00 to 2026-02-08 19:45:00
Number of Countries: 211


In [16]:
monthly_df = df.groupby(['year','month','country']).agg({
    'temperature_celsius':'mean',
    'precip_mm':'sum',
    'wind_kph':'mean'
}).reset_index()

monthly_df.head()


,year,month,country,temperature_celsius,precip_mm,wind_kph
0,2024,5,Afghanistan,20.305882,1.85,11.517647
1,2024,5,Albania,25.647059,5.02,13.076471
2,2024,5,Algeria,26.117647,0.01,22.114706
3,2024,5,Andorra,12.423529,5.98,9.876471
4,2024,5,Angola,29.088235,0.00,19.594118


In [17]:
df.to_csv("../data/processed/processed_weather_final.csv", index=False)